In [37]:
import torch
from einops import rearrange, einsum, reduce
from torch import nn

In [38]:
# ALl neural nets modules should inherent from nn.Module parent class -> inherits convenient methods such as: load_state_dict(), to(), get_parameters(), cpu(), cuda(), children(), bfloat16()...

# Implement a Linear Class (= "a Linear Module")
# y = xWT

class Linear(nn.Module): # Inherits nn.Module methods()
    def __init__(self, in_features, out_features, device=None, dtype=None):
        super().__init__()
        
        self.sigma = (2 / (in_features + out_features)) ** (1/2)

        self.W = nn.Parameter(nn.init.trunc_normal_(torch.empty(out_features, in_features, dtype=dtype, device=device), 
                                                                mean=0, std = self.sigma, 
                                                                a = -3 * self.sigma, b= 3 * self.sigma ))

    def forward(self, x: torch.tensor) -> torch.Tensor: # All nn.Module need to have a forward() method
        return einsum(x, self.W, '... in_feature , out_feature in_feature -> ... out_feature')

In [ ]:
# Create an embedding table Class
# Ounce again, every neural nets module should inherent nn.Module for convenient access to parent methods (.load_state_dict(), .Parameters(), .to()...)

class Embedding(nn.Module):

    def __init__(self, num_embeddings, embedding_dim, dtype=None, device=None):
        super().__init__()
        self.weights = nn.Parameter(nn.init.trunc_normal_(torch.empty(num_embeddings, embedding_dim, dtype=dtype, device=device), std=1, a=-3, b=3))

    def forward(self, x: torch.LongTensor) -> torch.Tensor: # (... T) -> (... T d_model)
        return self.weights[x] # shape: = x.shape + self.weights.shape[1:]